In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

if os.environ["OPENAI_API_KEY"] is None:
    raise ValueError("OPENAI_API_KEY environment variable not set.")
else:
    print("OPENAI_API_KEY environment variable is set.")


OPENAI_API_KEY environment variable is set.


## Process - Hierarical Process

Hierarical process add the manager agent:
 - Receives the high level goal
 - decides how to decompose the task
 - delegates the subtask to the worker agents
 - Validates their output


 

When to use Sequential vs Hierarical:


Sequential :
 - You know the exact steps
 - Predicatable, determinsitic
 - Fixed Pipeline ( Workflow A-> B -> C)

 Hierarical:
 - Steps are dynamic /Unknown
 - More autonomous
 - Task will be decomposed to subtask
 - Manager will validate it

In [6]:
from crewai import Agent,Task, Crew, Process

## worker agents

market_researcher = Agent(name="MarketResearcher", role="Market Researcher Specialist", goal="Conduct market research to identify potential opportunities and threats.",
                            backstory = """Expert at finding market data, competitor analysis and insdustry statistics. Always provived detailed  \
                            reports  in numbers and back the claims""",
                            verbose = True,
                            allow_delegation = False
                            )


data_analyst = Agent(name="DataAnalyst", role = "Data Analyst", goal = "Analyse the data and extract the business insights",
                    backstory = "Expert at turning the raw data into actionable business insights. Identifies trends, pattersn, risk and \
                        oppurtunities ",
                        verbose = True,
                        allow_delegation = False)


startergy_writer = Agent(name="StrategyWriter", role = "Strategy Report Writer", goal = "Synthesize the analysis into the executive ready report",
                        backstory = "Expert at writing clear and concise reports that distill complex data into actionable insights. Skilled at creating executive summaries and strategic recommendations.",
                        verbose = True,
                        allow_delegation = False)

In [10]:
stratergy_task = Task(name = "StrategyReportTask",
            description="""
                Create a comprehensive market strategy report using the research and analysis for the market {market} and company name {company}.

                The report must include:
                1. Executive Summary
                2. Market Overview
                3. Market Size & Growth
                4. Target Customers
                5. Competitive Analysis
                6. Product Positioning
                7. Go-To-Market Strategy
                8. Revenue Model
                9. Risks & Mitigation
                10. Implementation Roadmap

                Ensure:
                - Professional consulting style (McKinsey/BCG level)
                - Clear structure and headings
                - Actionable recommendations
                """,
                expected_output= """  Provide me the report in markdown format . Matching those 10 sections with clear heading.
                Use bullet points, tables and charts where appropriate. Total of 800-100 words""",
                agent=market_researcher ## manager will reassign


)

In [11]:
hierarichal_crew = Crew(name="Market Strategy Report Crew", agents=[market_researcher, data_analyst,startergy_writer], tasks=[stratergy_task], 
                        manager_llm="gpt-4o", process = Process.hierarchical, verbose=True)

In [12]:
hierarichal_crew

Crew(id=5f678e53-d0f9-445a-a0f7-9e38079be3b6, process=Process.hierarchical, number_of_agents=3, number_of_tasks=1)

In [13]:
result = hierarichal_crew.kickoff(inputs={"market": "Indian Rural banking","company":"A fintech and investment banking startup "})

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: Market Strategy Report Crew                                           │
│  ID: 5f678e53-d0f9-445a-a0f7-9e38079be3b6                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: StrategyReportTask                                                    │
│  ID: df89c7be-2fb1-4f3e-827

## Memory

# Short term memory -> It remebers context from current run
# Long term memory -> Store it in any data store
# Entity -> Extract specific things from the long term or short term memory ( basically stores the fact) 
# Contextutal -> relevant memories will be retived and passed to the LLM


In [2]:
from crewai import Agent, Task, Crew, Process

market_researcher = Agent(
    role="Research Specialist",
    goal="Research topics and build the knowledge base about {topic}",
    backstory="""You are an expert at finding relevant information, data and insights on the {topic}.
      You have access to a wide range of sources and are skilled at synthesizing information 
      into a comprehensive knowledge base.""",
    verbose=True,
    allow_delegation=False
    # ← No memory=True here; memory is a Crew-level setting
)

market_memory_task = Task(
    description="""
        Research and build a knowledge base about the topic {topic}.
        The knowledge base should include:
        1. Key Concepts and Definitions
        2. Historical Context and Evolution
        3. Current Trends and Developments
        4. Major Players and Stakeholders
        5. Challenges and Opportunities
        6. Future Outlook
        Ensure comprehensive coverage, clear organization, and 200-300 words overall.
        """,
    expected_output="A knowledge base in markdown format with clear headings for each section.",
    agent=market_researcher
)

memory_crew = Crew(
    agents=[market_researcher],
    tasks=[market_memory_task],
    process=Process.sequential,
    memory=True,   # ← This is what actually enables memory
    verbose=True
)

In [3]:
result1 = memory_crew.kickoff(inputs={"topic": "Artificial Intelligence in Healthcare"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b6f3092c-59aa-4c4e-b8e2-49515ecea375                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Research and build a knowledge base about the topic Artificial Intelligence in Healthcare.             │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│  ID: 8061afd6-450b-4762-833f-e81990349a73                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Research and build a knowledge base about the topic Artificial Intelligence in Healthcare.             │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['Artificial Intelligence in Healthcare key concepts', 'AI in healthcare definitions']}      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['Artificial Intelligence in Healthcare history', 'Evolution of AI in healthcare']}          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#7) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Output: No relevant memories found.                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Artificial Intelligence in Healthcare Knowledge Base                                                         │
│                                                                                                                 │
│  ## 1. Key Concepts and Definitions                                                                             │
│  Artificial Intelligence (AI) in Healthcare refers to the use of advanced algorithms and machine learning       │
│  techniques to analyze complex medical data. The goal is to improve diagnosis, treatment, patient outcomes,     │
│  and operational efficiency. Key concepts include machine learning, natural language processing, computer       │
│  vision, and robotics as applied to clinical and administrative healthcare tasks.                               │
│                                                                                                                 │
│  ## 2. Historical Context and Evolution                                                                         │
│  AI in healthcare has evolved from early rule-based expert systems in the 1970s to today's data-driven deep     │
│  learning models. The evolution accelerated with the advent of big data, improved computational power, and      │
│  access to vast amounts of electronic health records (EHRs). Milestones include early diagnostic support        │
│  systems, image analysis in radiology, and predictive analytics for patient risk.                               │
│                                                                                                                 │
│  ## 3. Current Trends and Developments                                                                          │
│  Current developments focus on precision medicine, AI-powered diagnostic tools, telemedicine integrations, and  │
│  natural language processing for clinical documentation. AI is increasingly used for drug discovery, robotic    │
│  surgery, and personalized treatment plans. Integration with wearable devices and remote monitoring is          │
│  enhancing chronic disease management.                                                                          │
│                                                                                                                 │
│  ## 4. Major Players and Stakeholders                                                                           │
│  Key stakeholders include technology companies (e.g., IBM Watson Health, Google Health, Microsoft Healthcare),  │
│  healthcare providers, research institutions, regulatory bodies, and patients. Collaborations between tech      │
│  firms and healthcare organizations are common, promoting innovation and deployment of AI solutions in          │
│  real-world settings.                                                                                           │
│                                                                                                                 │
│  ## 5. Challenges and Opportunities                                                                             │
│  Challenges encompass data privacy concerns, algorithmic bias, regulatory hurdles, and the need for             │
│  transparent AI models. Additionally, integration with existing healthcare systems and clinician acceptance     │
│  present barriers. Opportunities lie in improving diagn

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Research and build a knowledge base about the topic Artificial Intelligence in Healthcare.             │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b6f3092c-59aa-4c4e-b8e2-49515ecea375                                                                       │
│  Final Output: # Artificial Intelligence in Healthcare Knowledge Base                                           │
│                                                                                                                 │
│  ## 1. Key Concepts and Definitions                                                                             │
│  Artificial Intelligence (AI) in Healthcare refers to the use of advanced algorithms and machine learning       │
│  techniques to analyze complex medical data. The goal is to improve diagnosis, treatment, patient outcomes,     │
│  and operational efficiency. Key concepts include machine learning, natural language processing, computer       │
│  vision, and robotics as applied to clinical and administrative healthcare tasks.                               │
│                                                                                                                 │
│  ## 2. Historical Context and Evolution                                                                         │
│  AI in healthcare has evolved from early rule-based expert systems in the 1970s to today's data-driven deep     │
│  learning models. The evolution accelerated with the advent of big data, improved computational power, and      │
│  access to vast amounts of electronic health records (EHRs). Milestones include early diagnostic support        │
│  systems, image analysis in radiology, and predictive analytics for patient risk.                               │
│                                                                                                                 │
│  ## 3. Current Trends and Developments                                                                          │
│  Current developments focus on precision medicine, AI-powered diagnostic tools, telemedicine integrations, and  │
│  natural language processing for clinical documentation. AI is increasingly used for drug discovery, robotic    │
│  surgery, and personalized treatment plans. Integration with wearable devices and remote monitoring is          │
│  enhancing chronic disease management.                                                                          │
│                                                                                                                 │
│  ## 4. Major Players and Stakeholders                                                                           │
│  Key stakeholders include technology companies (e.g., IBM Watson Health, Google Health, Microsoft Healthcare),  │
│  healthcare providers, research institutions, regulatory bodies, and patients. Collaborations between tech      │
│  firms and healthcare organizations are common, promoting innovation and deployment of AI solutions in          │
│  real-world settings.                                                                                           │
│                                                                                                                 │
│  ## 5. Challenges and Opportunities                                                                             │
│  Challenges encompass data privacy concerns, algorithmic bias, regulatory hurdles, and the need for             │
│  transparent AI models. Additionally, integration with existing healthcare systems and clinician acceptance     │
│  present barriers. Opportunities lie in improving diag

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Check what memory attributes actually exist on your crew object
print(dir(memory_crew))

In [11]:
!pip3 install crewai

  Using cached chromadb-1.1.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.2 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
Using cached chromadb-1.1.1-cp39-abi3-macosx_11_0_arm64.whl (18.3 MB)
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s  0:00:00
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/4 [rich]
    Found existing installation: tokenizers 0.20.3━━━━━━━━━━━━━━━━ 1/4 [tokenizers]
    Uninstalling tokenizers-0.20.3:━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [tokenizers]
      Successfully uninstalled tokenizers-0.20.3━━━━━━━━━━━━━━ 1/4 [tokenizers]
  Attempting uninstall: openai90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [tokenizers]
    Found existing installation: openai 1.109.1━━━━━━━━━━━━━━━ 1/4 [tokenizers]
    Uninst

In [8]:
!pip3 install chromadb        # vector store (short-term + entity memory)
!pip3 install embedchain      # long-term memory backend

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/lancedb/__init__.py:289: UserWarning: lance is not fork-safe. If you are using multiprocessing, use spawn instead.
  warnings.warn(



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached chromadb-0.5.23-py3-none-any.whl.metadata (6.8 kB)
  Using cached langchain_openai-0.2.14-py3-none-any.whl.metadata (2.7 kB)
  Using cached langsmith-0.3.45-py3-none-any.whl.metadata (15 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
INFO: pip is looking at multiple versions of opentelemetry-instrumentation-fastapi to determine which version is compatible with other requirements. This could take a while.
  Using cached opentelemetry_instrumentation_fastapi-0.60b0-py3-none-any.whl.metadata (2.2 kB)
  Using cached opentelemetry_instrumentation_asgi-0.60b0-py3-none-any.whl.metadata (2.0 kB)
  Using cached opentelemetry_instrumentation-0.60b0-py3-none-any.whl.metadata (7.2 kB)
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation-

In [12]:

result1 = memory_crew.kickoff(inputs={"topic": "OpenAI"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b6f3092c-59aa-4c4e-b8e2-49515ecea375                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Research and build a knowledge base about the topic OpenAI.                                            │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│  ID: 8061afd6-450b-4762-833f-e81990349a73                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Research and build a knowledge base about the topic OpenAI.                                            │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#14) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['OpenAI key concepts definitions']}                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#15) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['OpenAI historical context evolution']}                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Tool search_memory executed with result: Found memories:
- (score=0.83) OpenAI is an artificial intelligence research organization focused on developing and promoting friendly AI for the benefit of humanity.
  categories: artificial intellig...
Tool search_memory executed with result: Found memories:
- (score=0.84) OpenAI is an artificial intelligence research organization focused on developing and promoting friendly AI for the benefit of humanity.
  categories: artificial intellig...
Tool search_memory executed with result: Found memories:
- (score=0.86) OpenAI is an artificial intelligence research organization focused on developing and promoting friendly AI for the benefit of humanity.
  categories: artificial intellig...
Tool search_memory executed with result: Found memories:
- (score=0.85) OpenAI is an artificial intelligence research organization focused on developing and promoting friendly AI for the benefit of humanity.
  categories: artificial intellig...
Tool search_memory e

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # OpenAI Knowledge Base                                                                                        │
│                                                                                                                 │
│  ## 1. Key Concepts and Definitions                                                                             │
│  OpenAI is an artificial intelligence research organization dedicated to developing and promoting friendly AI   │
│  for the benefit of humanity. Core concepts include machine learning, neural networks, natural language         │
│  processing (NLP), reinforcement learning, and generative models. OpenAI is well-known for its development of   │
│  large-scale language models such as the GPT series, which excel at language understanding and generation.      │
│                                                                                                                 │
│  ## 2. Historical Context and Evolution                                                                         │
│  OpenAI was founded in December 2015 by Elon Musk, Sam Altman, and others as a nonprofit entity focused on      │
│  open research to ensure artificial general intelligence (AGI) benefits all. In 2019 and 2020, OpenAI           │
│  developed the GPT-2 and GPT-3 models respectively, marking significant milestones in natural language          │
│  processing. To sustain growth and fund large-scale projects, OpenAI shifted to a capped-profit business        │
│  model.                                                                                                         │
│                                                                                                                 │
│  ## 3. Current Trends and Developments                                                                          │
│  OpenAI currently advances AI with models like GPT-4, which demonstrate enhanced language understanding and     │
│  creative abilities. The organization emphasizes practical AI applications such as conversational agents,       │
│  creative content generation, and coding assistants while prioritizing AI safety and ethical considerations.    │
│                                                                                                                 │
│  ## 4. Major Players and Stakeholders                                                                           │
│  Sam Altman serves as CEO and Mira Murati as CTO. Early involvement included Elon Musk. Major stakeholders      │
│  include corporate partners like Microsoft, which has invested billions, alongside investors, the AI research   │
│  community, and regulatory bodies focused on AI governance.                                                     │
│                                                                                                                 │
│  ## 5. Challenges and Opportunities                                                                             │
│  Challenges faced by OpenAI encompass AI safety, ethical use, bias in training data, privacy concerns,          │
│  regulatory hurdles, and high computational costs. Opportunities include transforming industries through        │
│  AI-powered automation, enhancing human creativity, and accelerating scientific research.                       │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Research and build a knowledge base about the topic OpenAI.                                            │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b6f3092c-59aa-4c4e-b8e2-49515ecea375                                                                       │
│  Final Output: # OpenAI Knowledge Base                                                                          │
│                                                                                                                 │
│  ## 1. Key Concepts and Definitions                                                                             │
│  OpenAI is an artificial intelligence research organization dedicated to developing and promoting friendly AI   │
│  for the benefit of humanity. Core concepts include machine learning, neural networks, natural language         │
│  processing (NLP), reinforcement learning, and generative models. OpenAI is well-known for its development of   │
│  large-scale language models such as the GPT series, which excel at language understanding and generation.      │
│                                                                                                                 │
│  ## 2. Historical Context and Evolution                                                                         │
│  OpenAI was founded in December 2015 by Elon Musk, Sam Altman, and others as a nonprofit entity focused on      │
│  open research to ensure artificial general intelligence (AGI) benefits all. In 2019 and 2020, OpenAI           │
│  developed the GPT-2 and GPT-3 models respectively, marking significant milestones in natural language          │
│  processing. To sustain growth and fund large-scale projects, OpenAI shifted to a capped-profit business        │
│  model.                                                                                                         │
│                                                                                                                 │
│  ## 3. Current Trends and Developments                                                                          │
│  OpenAI currently advances AI with models like GPT-4, which demonstrate enhanced language understanding and     │
│  creative abilities. The organization emphasizes practical AI applications such as conversational agents,       │
│  creative content generation, and coding assistants while prioritizing AI safety and ethical considerations.    │
│                                                                                                                 │
│  ## 4. Major Players and Stakeholders                                                                           │
│  Sam Altman serves as CEO and Mira Murati as CTO. Early involvement included Elon Musk. Major stakeholders      │
│  include corporate partners like Microsoft, which has invested billions, alongside investors, the AI research   │
│  community, and regulatory bodies focused on AI governance.                                                     │
│                                                                                                                 │
│  ## 5. Challenges and Opportunities                                                                             │
│  Challenges faced by OpenAI encompass AI safety, ethical use, bias in training data, privacy concerns,          │
│  regulatory hurdles, and high computational costs. Opportunities include transforming industries through        │
│  AI-powered automation, enhancing human creativity, and accelerating scientific research.                       │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [13]:
result4 = memory_crew.kickoff(inputs={"topic": "Anthropic AI"})
print(result4)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b6f3092c-59aa-4c4e-b8e2-49515ecea375                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Research and build a knowledge base about the topic Anthropic AI.                                      │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│  ID: 8061afd6-450b-4762-833f-e81990349a73                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Research and build a knowledge base about the topic Anthropic AI.                                      │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#20) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['Anthropic AI key concepts', 'Anthropic AI definitions', 'Anthropic AI history',            │
│  'Anthropic AI evolution', 'Anthropic AI current trends', 'Anthropic AI developments', 'Anthropic AI major ...  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_memory executed with result: Found memories:
- (score=0.85) Key concepts in AI in Healthcare include machine learning, natural language processing, computer vision, and robotics.
  categories: AI, Healthcare, Technology
  entitie...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Anthropic AI Knowledge Base                                                                                  │
│                                                                                                                 │
│  ## 1. Key Concepts and Definitions                                                                             │
│  Anthropic AI is an artificial intelligence research company focused on developing reliable, interpretable,     │
│  and steerable AI systems. Core concepts include AI safety, alignment, transparency, and ethical                │
│  considerations in AI development. The company emphasizes building AI that can be controlled and understood to  │
│  avoid unintended harmful consequences.                                                                         │
│                                                                                                                 │
│  ## 2. Historical Context and Evolution                                                                         │
│  Anthropic AI was founded in 2021 by former OpenAI researchers with the mission to address the challenges of    │
│  AI safety and ethical AI development. Its establishment reflects increasing concerns about the risks           │
│  associated with powerful AI systems and the need for industry-wide leadership on AI alignment.                 │
│                                                                                                                 │
│  ## 3. Current Trends and Developments                                                                          │
│  Anthropic AI is actively researching scalable safety techniques and developing large language models with      │
│  built-in safety measures. Their approach focuses on making AI behavior more predictable and controllable       │
│  through innovations in interpretability and reinforcement learning from human feedback.                        │
│                                                                                                                 │
│  ## 4. Major Players and Stakeholders                                                                           │
│  Key figures include co-founders Dario Amodei, former VP of Research at OpenAI, and Daniela Amodei.             │
│  Stakeholders include AI researchers, corporate partners, investors, and policymakers interested in             │
│  responsible AI development. The company collaborates with the broader AI community to foster best practices    │
│  in AI safety.                                                                                                  │
│                                                                                                                 │
│  ## 5. Challenges and Opportunities                                                                             │
│  Challenges faced by Anthropic AI include ensuring AI alignment as models grow more complex, mitigating         │
│  biases, addressing regulatory and ethical concerns, and managing computational costs. Opportunities lie in     │
│  pioneering safer AI architectures, influencing AI governance, and providing trustworthy AI solutions across    │
│  industries.                                                                                                    │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Research and build a knowledge base about the topic Anthropic AI.                                      │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b6f3092c-59aa-4c4e-b8e2-49515ecea375                                                                       │
│  Final Output: # Anthropic AI Knowledge Base                                                                    │
│                                                                                                                 │
│  ## 1. Key Concepts and Definitions                                                                             │
│  Anthropic AI is an artificial intelligence research company focused on developing reliable, interpretable,     │
│  and steerable AI systems. Core concepts include AI safety, alignment, transparency, and ethical                │
│  considerations in AI development. The company emphasizes building AI that can be controlled and understood to  │
│  avoid unintended harmful consequences.                                                                         │
│                                                                                                                 │
│  ## 2. Historical Context and Evolution                                                                         │
│  Anthropic AI was founded in 2021 by former OpenAI researchers with the mission to address the challenges of    │
│  AI safety and ethical AI development. Its establishment reflects increasing concerns about the risks           │
│  associated with powerful AI systems and the need for industry-wide leadership on AI alignment.                 │
│                                                                                                                 │
│  ## 3. Current Trends and Developments                                                                          │
│  Anthropic AI is actively researching scalable safety techniques and developing large language models with      │
│  built-in safety measures. Their approach focuses on making AI behavior more predictable and controllable       │
│  through innovations in interpretability and reinforcement learning from human feedback.                        │
│                                                                                                                 │
│  ## 4. Major Players and Stakeholders                                                                           │
│  Key figures include co-founders Dario Amodei, former VP of Research at OpenAI, and Daniela Amodei.             │
│  Stakeholders include AI researchers, corporate partners, investors, and policymakers interested in             │
│  responsible AI development. The company collaborates with the broader AI community to foster best practices    │
│  in AI safety.                                                                                                  │
│                                                                                                                 │
│  ## 5. Challenges and Opportunities                                                                             │
│  Challenges faced by Anthropic AI include ensuring AI alignment as models grow more complex, mitigating         │
│  biases, addressing regulatory and ethical concerns, and managing computational costs. Opportunities lie in     │
│  pioneering safer AI architectures, influencing AI governance, and providing trustworthy AI solutions across    │
│  industries.                                                                                                    │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Anthropic AI Knowledge Base

## 1. Key Concepts and Definitions
Anthropic AI is an artificial intelligence research company focused on developing reliable, interpretable, and steerable AI systems. Core concepts include AI safety, alignment, transparency, and ethical considerations in AI development. The company emphasizes building AI that can be controlled and understood to avoid unintended harmful consequences.

## 2. Historical Context and Evolution
Anthropic AI was founded in 2021 by former OpenAI researchers with the mission to address the challenges of AI safety and ethical AI development. Its establishment reflects increasing concerns about the risks associated with powerful AI systems and the need for industry-wide leadership on AI alignment.

## 3. Current Trends and Developments
Anthropic AI is actively researching scalable safety techniques and developing large language models with built-in safety measures. Their approach focuses on making AI behavior more predictable and c

In [16]:
result5 = memory_crew.kickoff(inputs={"topic": "Open AI vs Anthorphci AI"})
print(result5)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b6f3092c-59aa-4c4e-b8e2-49515ecea375                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Research and build a knowledge base about the topic Open AI vs Anthorphci AI.                          │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│  ID: 8061afd6-450b-4762-833f-e81990349a73                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Research and build a knowledge base about the topic Open AI vs Anthorphci AI.                          │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#21) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['OpenAI key concepts', 'OpenAI definitions']}                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#22) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['Anthropic AI key concepts', 'Anthropic AI definitions']}                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # OpenAI vs Anthropic AI Knowledge Base                                                                        │
│                                                                                                                 │
│  ## 1. Key Concepts and Definitions                                                                             │
│  **OpenAI** is an artificial intelligence research organization dedicated to developing and promoting friendly  │
│  AI for the benefit of humanity. It focuses on broad AI capabilities, including natural language processing,    │
│  reinforcement learning, and large-scale models like GPT-4.                                                     │
│                                                                                                                 │
│  **Anthropic AI** is an AI research company focused on developing reliable, interpretable, and steerable AI     │
│  systems. It emphasizes AI safety, alignment, transparency, and ethical considerations, aiming to create AI     │
│  technologies that align with human values and can be controlled safely.                                        │
│                                                                                                                 │
│  ## 2. Historical Context and Evolution                                                                         │
│  OpenAI was founded in December 2015 by Elon Musk, Sam Altman, and others as a nonprofit aimed at open AI       │
│  research for global benefit. Over time, it shifted to a capped-profit model to fund large projects like GPT-2  │
│  and GPT-3.                                                                                                     │
│                                                                                                                 │
│  Anthropic AI was founded in 2021 by former OpenAI researchers Dario Amodei and Daniela Amodei to address       │
│  concerns about AI safety and ethical development. The company’s founding reflects growing emphasis on AI       │
│  alignment and risk mitigation.                                                                                 │
│                                                                                                                 │
│  ## 3. Current Trends and Developments                                                                          │
│  OpenAI leads in developing powerful AI models such as GPT-4 with advanced language understanding and creative  │
│  capabilities. It focuses on AI-powered automation, enhancing human creativity, and accelerating scientific     │
│  discovery.                                                                                                     │
│                                                                                                                 │
│  Anthropic AI is engaged in research to implement scalable AI safety solutions, focusing on improved            │
│  interpretability, reinforcement learning from human feedback, and controllable AI behavior to reduce           │
│  unintended risks.                                                                                              │
│                                                                                                                 │
│  ## 4. Major Players and Stakeholders                  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Research and build a knowledge base about the topic Open AI vs Anthorphci AI.                          │
│          The knowledge base should include:                                                                     │
│          1. Key Concepts and Definitions                                                                        │
│          2. Historical Context and Evolution                                                                    │
│          3. Current Trends and Developments                                                                     │
│          4. Major Players and Stakeholders                                                                      │
│          5. Challenges and Opportunities                                                                        │
│          6. Future Outlook                                                                                      │
│          Ensure comprehensive coverage, clear organization, and 200-300 words overall.                          │
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b6f3092c-59aa-4c4e-b8e2-49515ecea375                                                                       │
│  Final Output: # OpenAI vs Anthropic AI Knowledge Base                                                          │
│                                                                                                                 │
│  ## 1. Key Concepts and Definitions                                                                             │
│  **OpenAI** is an artificial intelligence research organization dedicated to developing and promoting friendly  │
│  AI for the benefit of humanity. It focuses on broad AI capabilities, including natural language processing,    │
│  reinforcement learning, and large-scale models like GPT-4.                                                     │
│                                                                                                                 │
│  **Anthropic AI** is an AI research company focused on developing reliable, interpretable, and steerable AI     │
│  systems. It emphasizes AI safety, alignment, transparency, and ethical considerations, aiming to create AI     │
│  technologies that align with human values and can be controlled safely.                                        │
│                                                                                                                 │
│  ## 2. Historical Context and Evolution                                                                         │
│  OpenAI was founded in December 2015 by Elon Musk, Sam Altman, and others as a nonprofit aimed at open AI       │
│  research for global benefit. Over time, it shifted to a capped-profit model to fund large projects like GPT-2  │
│  and GPT-3.                                                                                                     │
│                                                                                                                 │
│  Anthropic AI was founded in 2021 by former OpenAI researchers Dario Amodei and Daniela Amodei to address       │
│  concerns about AI safety and ethical development. The company’s founding reflects growing emphasis on AI       │
│  alignment and risk mitigation.                                                                                 │
│                                                                                                                 │
│  ## 3. Current Trends and Developments                                                                          │
│  OpenAI leads in developing powerful AI models such as GPT-4 with advanced language understanding and creative  │
│  capabilities. It focuses on AI-powered automation, enhancing human creativity, and accelerating scientific     │
│  discovery.                                                                                                     │
│                                                                                                                 │
│  Anthropic AI is engaged in research to implement scalable AI safety solutions, focusing on improved            │
│  interpretability, reinforcement learning from human feedback, and controllable AI behavior to reduce           │
│  unintended risks.                                                                                              │
│                                                                                                                 │
│  ## 4. Major Players and Stakeholders                 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# OpenAI vs Anthropic AI Knowledge Base

## 1. Key Concepts and Definitions
**OpenAI** is an artificial intelligence research organization dedicated to developing and promoting friendly AI for the benefit of humanity. It focuses on broad AI capabilities, including natural language processing, reinforcement learning, and large-scale models like GPT-4.

**Anthropic AI** is an AI research company focused on developing reliable, interpretable, and steerable AI systems. It emphasizes AI safety, alignment, transparency, and ethical considerations, aiming to create AI technologies that align with human values and can be controlled safely.

## 2. Historical Context and Evolution
OpenAI was founded in December 2015 by Elon Musk, Sam Altman, and others as a nonprofit aimed at open AI research for global benefit. Over time, it shifted to a capped-profit model to fund large projects like GPT-2 and GPT-3.

Anthropic AI was founded in 2021 by former OpenAI researchers Dario Amodei and Daniela Amodei

## Flows

Crews are autonomous -> Agent execute based on collaboration


Flows are structured and event driven -> Agents executon on conditional and routers

In [ ]:
Crews are autonomous -> Agent execute based on collaboration
Flows are structured and event driven -> Agents executon on conditional and routers

@start() -> this is where your flow begins

1. only one method should have @start()

2. This method runs first when the flow the triggered



@listen(method_name) -> Dependency
def clean_date(self,data):
    data.strip()
This method will runs after the given method finishes

1. The output of the previous method will be passed as input to this method.

@router(method_name) -> This is used in decision making flows.



In [17]:
# @router(analyze_data):
# def route_decision(self, result):
#     if x in result:
#         return "success_handler"
#     else:
#       return "error_handler


# @listen(success_handler)
# def handle_sucess():

# ....


# @listen(error_handler)
# def handle_error():

# ...

##. Use case
1. Generate content
2. score the quality content
3. if score is high -> then publish it
4. Else -> send for revision or retry

In [ ]:
from crewai.flow.flow import Flow, router, listen, start
from crewai import Agent, Task, Crew, Process,LLM
from pydantic import BaseModel

class QAPipelineInput(BaseModel):
    topic : str =''
    draft_content : str =''
    quality_score :float = 0.0
    feedback : str =''
    final_content : str =''
    revision_count : int = 0
    


llm = LLM(model="gpt-4o", temperature=0.7)


In [4]:
writer_agent = Agent(name="WriterAgent", role="Content Writer", goal="Generate a draft content based on the topic", llm=llm,
                     backstory = "You are a skilled content writer with a keen eye for detail and a passion for creating engaging narratives.", verbose=True, allow_delegation=False)

reviewer_agent = Agent(name="ReviewerAgent", role="Content Reviewer", goal="Review the draft content and provide feedback for improvement", llm=llm,
                       backstory = "You are a meticulous content reviewer with a deep understanding of quality standards and a talent for identifying areas for improvement.", verbose=True, allow_delegation=False)

editor_agent = Agent(name="EditorAgent", role="Content Editor", goal="Edit the draft content based on the feedback and improve the quality score", llm=llm,
                     backstory = "You are an experienced content editor with a keen sense of style and structure, dedicated to enhancing the overall quality of written work.", verbose=True, allow_delegation=False)

In [7]:
class QAPipeline(Flow[QAPipelineInput]):

    @start()
    def generate_draft(self):
        print(f"Generating the draft for the topic: {self.state.topic}")
        task = Task(
            description=f"""
                Generate a draft article based on the topic: {self.state.topic}.
                Ensure the content is engaging, well-structured, and informative.
            """,
            expected_output="Well structured article with max 500 words, proper format and sections.",
            agent=writer_agent
        )
        crew = Crew(agents=[writer_agent], tasks=[task])
        result = crew.kickoff()
        self.state.draft_content = str(result)
        print(f"Draft content generated:\n{self.state.draft_content}")
        return self.state.draft_content

    @listen(generate_draft)
    def score_quality(self):
        print("Scoring the quality of the draft content...")
        task = Task(
            description=f"""
                Score the quality of the following content on a scale of 1 to 10.
                Consider clarity, coherence, engagement, and informativeness.

                Content:
                {self.state.draft_content}

                Respond with ONLY a number between 1 and 10.
            """,
            expected_output="A single numerical score between 1 and 10.",
            agent=reviewer_agent
        )
        crew = Crew(agents=[reviewer_agent], tasks=[task])
        result = crew.kickoff()
        self.state.quality_score = float(str(result).strip())
        print(f"Quality score assigned: {self.state.quality_score}")
        return self.state.quality_score

    @router(score_quality)
    def route_by_quality_score(self):
        if self.state.quality_score >= 7.0 or self.state.revision_count >= 3:
            return "publish"
        else:
            return "revise"

    @listen('revise')
    def revise_content(self):
        print("Revising the draft content based on feedback...")
        task = Task(
            description=f"""
                Revise the following draft content to improve its quality.
                Focus on clarity, structure, coherence, and engagement.

                Current content:
                {self.state.draft_content}
            """,
            expected_output="A revised version of the draft content with improved quality.",
            agent=editor_agent
        )
        crew = Crew(agents=[editor_agent], tasks=[task])
        result = crew.kickoff()
        self.state.draft_content = str(result)
        self.state.revision_count += 1
        print(f"Draft revised. Revision count: {self.state.revision_count}")
        return self.state.draft_content

    @listen('publish')
    def publish_content(self):
        print(f"Quality score: {self.state.quality_score}. Publishing content.")
        self.state.final_content = self.state.draft_content
        print(f"Final content published: {self.state.final_content}")
        return self.state.final_content




In [8]:
qa_flow = QAPipeline()
result = qa_flow.kickoff(inputs={"topic": "The impact of Artificial Intelligence on modern education"})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: QAPipeline                                                                                               │
│  ID: c5d72be1-95db-4782-be6d-821cc07994be                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: QAPipeline                                                                                               │
│  ID: c5d72be1-95db-4782-be6d-821cc07994be                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: c5d72be1-95db-4782-be6d-821cc07994be

Generating the draft for the topic: The impact of Artificial Intelligence on modern education


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│                  Generate a draft article based on the topic: The impact of Artificial Intelligence on modern   │
│  education.                                                                                                     │
│                  Ensure the content is engaging, well-structured, and informative.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Draft content generated:
**The Impact of Artificial Intelligence on Modern Education**

Artificial Intelligence (AI) has rapidly become a pivotal force in reshaping modern education. Its integration into educational systems worldwide has introduced innovative methods for learning and teaching, enhancing educational experiences and outcomes. This article delves into the impact of AI on education, examining its advantages, challenges, and the future it promises.

**Revolutionizing Personalized Learning**

AI's most transformative effect on education is perhaps its ability to provide personalized learning experiences. Through adaptive learning technologies, AI can analyze a student's strengths and weaknesses, customizing the curriculum to fit individual learning styles. Platforms like Coursera and Khan Academy have harnessed AI to offer tailored content that meets each student's pace and proficiency level. This personalization not only fosters improved learning outcomes but also keeps stu

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Reviewer                                                                                        │
│                                                                                                                 │
│  Task:                                                                                                          │
│                  Score the quality of the following content on a scale of 1 to 10.                              │
│                  Consider clarity, coherence, engagement, and informativeness.                                  │
│                                                                                                                 │
│                  Content:                                                                                       │
│                  **The Impact of Artificial Intelligence on Modern Education**                                  │
│                                                                                                                 │
│  Artificial Intelligence (AI) has rapidly become a pivotal force in reshaping modern education. Its             │
│  integration into educational systems worldwide has introduced innovative methods for learning and teaching,    │
│  enhancing educational experiences and outcomes. This article delves into the impact of AI on education,        │
│  examining its advantages, challenges, and the future it promises.                                              │
│                                                                                                                 │
│  **Revolutionizing Personalized Learning**                                                                      │
│                                                                                                                 │
│  AI's most transformative effect on education is perhaps its ability to provide personalized learning           │
│  experiences. Through adaptive learning technologies, AI can analyze a student's strengths and weaknesses,      │
│  customizing the curriculum to fit individual learning styles. Platforms like Coursera and Khan Academy have    │
│  harnessed AI to offer tailored content that meets each student's pace and proficiency level. This              │
│  personalization not only fosters improved learning outcomes but also keeps students engaged by addressing      │
│  their unique needs.                                                                                            │
│                                                                                                                 │
│  **Empowering Educators with Advanced Tools**                                                                   │
│                                                                                                                 │
│  AI is also revolutionizing the role of educators by providing powerful tools that enhance teaching             │
│  efficiency. Automated grading systems, such as those used by platforms like Gradescope, significantly reduce   │
│  the time teachers spend on administrative tasks, allowing them to focus more on student interaction and        │
│  lesson planning. Furthermore, AI-driven analytics offer insights into student performance, helping educators   │
│  identify struggling students and tailor their support accordingly.                                             │
│                                                                                                                 │
│  **Enhancing Accessibility and Inclusivity**                                                                    │
│                                                        

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Reviewer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  8                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Quality score assigned: 8.0
Quality score: 8.0. Publishing content.
Final content published: **The Impact of Artificial Intelligence on Modern Education**

Artificial Intelligence (AI) has rapidly become a pivotal force in reshaping modern education. Its integration into educational systems worldwide has introduced innovative methods for learning and teaching, enhancing educational experiences and outcomes. This article delves into the impact of AI on education, examining its advantages, challenges, and the future it promises.

**Revolutionizing Personalized Learning**

AI's most transformative effect on education is perhaps its ability to provide personalized learning experiences. Through adaptive learning technologies, AI can analyze a student's strengths and weaknesses, customizing the curriculum to fit individual learning styles. Platforms like Coursera and Khan Academy have harnessed AI to offer tailored content that meets each student's pace and proficiency level. This personaliz

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: QAPipeline                                                                                               │
│  ID: c5d72be1-95db-4782-be6d-821cc07994be                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯